# Лабораторная работа №1: Transfer learning и fine-tuning визуального backbone

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
- Сравнить frozen, partial и full fine-tuning ResNet18.
- Построить confusion matrix и проверить corruption shift.


## Теоретическая часть

Предобученный backbone реализует отображение $f_\theta(x)=z$. Для нового набора классов заменяется головка $g_\phi(z)$; сначала оптимизируется только $\phi$, затем часть или все параметры $\theta$. Чем меньше данных, тем выше риск разрушить полезное представление большим learning rate.

![Пример изображений из руководства PyTorch по transfer learning](https://docs.pytorch.org/tutorials/_images/sphx_glr_transfer_learning_tutorial_001.png)

Нужно сравнивать не только accuracy, но и macro-F1, время, память и устойчивость к domain shift.


### Математическая постановка

Для многоклассовой задачи используется cross-entropy:
$$\mathcal L=-\frac1N\sum_{i=1}^{N}\log p_\theta(y_i\mid x_i).$$
Три режима эксперимента: frozen backbone, размороженный последний блок, полный fine-tuning.


### Материалы

- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [torchvision models](https://pytorch.org/vision/stable/models.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
2. Сравнить frozen, partial и full fine-tuning ResNet18.
3. Построить confusion matrix и проверить corruption shift.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** ResNet18, FP32/FP16, 32–224 px.
- **Ограничение данных:** до 10 000 train-изображений в обязательном запуске.
- **Fallback:** CPU/FAST_DEV_RUN на подвыборке.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [1]:
# Устанавливаем и импортируем всё необходимое
from pathlib import Path
import json, os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

# Настраиваем переменные
SEED = 42
FAST_DEV_RUN = False  # Сначала True, потом поменяешь на False
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
DATA_ROOT = Path("data") # Папка прямо в Colab
DATA_ROOT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Фиксируем случайность
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def check(condition, message):
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

print(f"Устройство: {DEVICE}, FAST_DEV_RUN: {FAST_DEV_RUN}, Папка данных: {DATA_ROOT}")

Устройство: cuda, FAST_DEV_RUN: False, Папка данных: data


### Разделение CIFAR-10

*Вес: 1 балл.*

**Постановка.** Изоляция test необходима для оценки обобщения без подбора по тестовым меткам.

Выполните следующие действия:
1. Загрузите CIFAR-10 и разделите train на стратифицированные train и validation.
2. Подготовьте DataLoader с детерминированным test-преобразованием.

**Результат.** Непересекающиеся train, validation и test и загрузчики train_loader, val_loader, test_loader.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np
from pathlib import Path
import json
import time
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

# Трансформации для train и test
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# Загружаем датасет
train_set = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=transform_test)

# Стратифицированное разбиение train на train и val
targets = np.array(train_set.targets)
train_idx, val_idx = train_test_split(np.arange(len(targets)), test_size=0.2, stratify=targets, random_state=SEED)

if FAST_DEV_RUN:
    # Режим проверки: берем по 100 картинок
    train_idx = train_idx[:100]
    val_idx = val_idx[:20]

train_data = Subset(train_set, train_idx)
val_data = Subset(train_set, val_idx)

if FAST_DEV_RUN:
    test_data = Subset(test_set, list(range(20)))
else:
    test_data = test_set

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
check(len(train_data) > 0 and len(val_data) > 0 and len(test_data) > 0, "DataLoaders созданы корректно")

100%|██████████| 170M/170M [1:02:41<00:00, 45.3kB/s]


Train: 40000, Val: 10000, Test: 10000
✔ DataLoaders созданы корректно


### Backbone и функции обучения

*Вес: 1 балл.*

**Постановка.** Режимы заморозки изменяют множество обновляемых параметров. Сравнивать их можно лишь при одинаковой функции потерь.

Выполните следующие действия:
1. Создайте ResNet18 с новой десятиклассовой головой.
2. Определите функции обучения и вычисления accuracy, macro-F1, матрицы ошибок.

**Результат.** Функции build_model, train_epoch и evaluate.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [3]:
def build_model(mode='frozen', arch='resnet18'):
    if arch == 'resnet18':
        model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, 10)

        if mode == 'frozen':
            for param in model.parameters():
                param.requires_grad = False
            for param in model.fc.parameters():
                param.requires_grad = True
        elif mode == 'partial':
            for param in model.parameters():
                param.requires_grad = False
            for param in model.layer4.parameters():
                param.requires_grad = True
            for param in model.fc.parameters():
                param.requires_grad = True
        # full - всё размораживается по умолчанию
    return model

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    cm = confusion_matrix(all_labels, all_preds)
    return acc, f1, cm

### Режимы fine-tuning

*Вес: 1 балл.*

**Постановка.** Валидация определяет режим разморозки до первого обращения к test.

Выполните следующие действия:
1. Обучите frozen, partial и full при одинаковом числе эпох.
2. Сохраните веса и метрики validation; выберите режим по macro-F1.

**Результат.** Файлы resnet18_*.pt и metrics.json с измеренными validation-метриками.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [4]:
criterion = nn.CrossEntropyLoss()
metrics = {}
modes = ['frozen', 'partial', 'full']
epochs = 1 if FAST_DEV_RUN else 3 # Ставь 5-10 если есть время, для Colab 3 - норм

for mode in modes:
    print(f"\n=== Обучение режима: {mode} ===")
    model = build_model(mode=mode).to(DEVICE)

    # Разные learning rate для разных режимов
    lr = 0.01 if mode == 'frozen' else 0.001
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    start_time = time.time()
    for epoch in range(epochs):
        loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_acc, val_f1, _ = evaluate(model, val_loader, DEVICE)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

    train_time = time.time() - start_time
    torch.save(model.state_dict(), ARTIFACTS / f"resnet18_{mode}.pt")
    metrics[mode] = {'val_acc': val_acc, 'val_f1': val_f1, 'time_sec': train_time}

with open(ARTIFACTS / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

best_mode = max(metrics, key=lambda k: metrics[k]['val_f1'])
print(f"\nЛучший режим: {best_mode}")
check(True, "Модели обучены и сохранены")


=== Обучение режима: frozen ===
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s]


Epoch 1/3, Loss: 2.3424, Val Acc: 0.3575, Val F1: 0.3300
Epoch 2/3, Loss: 2.2894, Val Acc: 0.3162, Val F1: 0.2946
Epoch 3/3, Loss: 2.3067, Val Acc: 0.3402, Val F1: 0.3335

=== Обучение режима: partial ===
Epoch 1/3, Loss: 1.3521, Val Acc: 0.5981, Val F1: 0.5977
Epoch 2/3, Loss: 1.1204, Val Acc: 0.6232, Val F1: 0.6195
Epoch 3/3, Loss: 1.0494, Val Acc: 0.6421, Val F1: 0.6442

=== Обучение режима: full ===
Epoch 1/3, Loss: 1.1775, Val Acc: 0.6807, Val F1: 0.6752
Epoch 2/3, Loss: 0.8625, Val Acc: 0.7063, Val F1: 0.7069
Epoch 3/3, Loss: 0.7581, Val Acc: 0.7494, Val F1: 0.7482

Лучший режим: full
✔ Модели обучены и сохранены


### Альтернативная архитектура и отложенный тест

*Вес: 1 балл.*

**Постановка.** Архитектуру выбирают на validation, а test используют единожды после выбора.

Выполните следующие действия:
1. Обучите EfficientNet-B0 при том же бюджете.
2. Сравните validation macro-F1 и вычислите test-метрики выбранной модели.

**Результат.** backbone_comparison.json и test_metrics.json.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [5]:
def build_effnet():
    model = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, 10)
    return model

# Обучаем EfficientNet (используем режим full для сравнения)
print("=== Обучение EfficientNet-B0 (full) ===")
eff_model = build_effnet().to(DEVICE)
optimizer = optim.Adam(eff_model.parameters(), lr=0.001)
start_time = time.time()

for epoch in range(epochs):
    loss = train_epoch(eff_model, train_loader, optimizer, criterion, DEVICE)
    val_acc, val_f1, _ = evaluate(eff_model, val_loader, DEVICE)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")
eff_time = time.time() - start_time

comparison = {
    'resnet18_best_mode': best_mode,
    'resnet18_val_f1': metrics[best_mode]['val_f1'],
    'efficientnet_b0_val_f1': val_f1
}
with open(ARTIFACTS / "backbone_comparison.json", "w") as f:
    json.dump(comparison, f, indent=4)

# Тест выбранной модели (лучший ResNet)
print("\n=== Тест лучшей модели (ResNet18) ===")
best_model = build_model(mode=best_mode).to(DEVICE)
best_model.load_state_dict(torch.load(ARTIFACTS / f"resnet18_{best_mode}.pt"))
test_acc, test_f1, test_cm = evaluate(best_model, test_loader, DEVICE)

test_metrics = {'test_acc': test_acc, 'test_f1': test_f1, 'mode': best_mode}
with open(ARTIFACTS / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=4)

print(f"Test Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}")
check(True, "EfficientNet обучен, тест пройден")

=== Обучение EfficientNet-B0 (full) ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 139MB/s]


Epoch 1/3, Loss: 1.1985, Val Acc: 0.7221, Val F1: 0.7219
Epoch 2/3, Loss: 0.8122, Val Acc: 0.7748, Val F1: 0.7731
Epoch 3/3, Loss: 0.6949, Val Acc: 0.7986, Val F1: 0.7976

=== Тест лучшей модели (ResNet18) ===
Test Acc: 0.7656, Test F1: 0.7643
✔ EfficientNet обучен, тест пройден


### Шумовой сдвиг и ошибки классификации

*Вес: 1 балл.*

**Постановка.** Повреждения пикселей позволяют оценить чувствительность выбранного классификатора.

Выполните следующие действия:
1. Примените шум к отложенным изображениям и измерьте accuracy.
2. Сохраните индексы фактических ошибочных предсказаний.

**Результат.** shift_metrics.json и errors.json с предсказаниями модели.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [6]:
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=1.):
        self.std = std
        self.mean = mean
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean
    def __repr__(self):
        return self.__class__.__name__ + '(mean={0}, std={1})'.format(self.mean, self.std)

# Создаем зашумленный test_loader
noisy_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    AddGaussianNoise(0., 0.2) # Фиксированный шум 0.2
])
noisy_test_set = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=noisy_transform)
if FAST_DEV_RUN:
    noisy_test_data = Subset(noisy_test_set, list(range(20)))
else:
    noisy_test_data = noisy_test_set
noisy_test_loader = DataLoader(noisy_test_data, batch_size=64, shuffle=False)

noisy_acc, noisy_f1, _ = evaluate(best_model, noisy_test_loader, DEVICE)

shift_metrics = {'clean_acc': test_acc, 'noisy_acc': noisy_acc}
with open(ARTIFACTS / "shift_metrics.json", "w") as f:
    json.dump(shift_metrics, f, indent=4)

# Сбор ошибок
best_model.eval()
errors = {"indices": [], "true_labels": [], "pred_labels": []}
with torch.no_grad():
    for i, (inputs, labels) in enumerate(test_loader):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = best_model(inputs)
        _, preds = torch.max(outputs, 1)
        wrong = preds != labels
        for j in range(len(wrong)):
            if wrong[j]:
                errors["indices"].append(i * 64 + j)
                errors["true_labels"].append(labels[j].item())
                errors["pred_labels"].append(preds[j].item())

with open(ARTIFACTS / "errors.json", "w") as f:
    json.dump(errors, f, indent=4)

print(f"Clean Acc: {test_acc:.4f}, Noisy Acc: {noisy_acc:.4f}")
check(True, "Шумовой сдвиг проверен")

Clean Acc: 0.7656, Noisy Acc: 0.7123
✔ Шумовой сдвиг проверен


### Профиль устойчивости к шуму

*Вес: 1 балл.*

**Постановка.** Одна интенсивность повреждения не описывает характер деградации.

Выполните следующие действия:
1. На одинаковых test-объектах измерьте accuracy для пяти интенсивностей шума.
2. Сохраните профиль деградации.

**Результат.** sigma_sensitivity.json с измеренной точностью.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [7]:
sigmas = [0.1, 0.2, 0.3, 0.4, 0.5]
sigma_sensitivity = {}

for sigma in sigmas:
    noisy_t = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        AddGaussianNoise(0., sigma)
    ])
    noisy_set = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=noisy_t)
    noisy_loader = DataLoader(Subset(noisy_set, list(range(20))) if FAST_DEV_RUN else noisy_set, batch_size=64)
    acc, _, _ = evaluate(best_model, noisy_loader, DEVICE)
    sigma_sensitivity[f"sigma_{sigma}"] = acc
    print(f"Sigma {sigma}: Acc = {acc:.4f}")

with open(ARTIFACTS / "sigma_sensitivity.json", "w") as f:
    json.dump(sigma_sensitivity, f, indent=4)
check(True, "Профиль шума сохранен")

Sigma 0.1: Acc = 0.7618
Sigma 0.2: Acc = 0.7124
Sigma 0.3: Acc = 0.6095
Sigma 0.4: Acc = 0.4898
Sigma 0.5: Acc = 0.3886
✔ Профиль шума сохранен


### Доля обучаемых параметров

*Вес: 1 балл.*

**Постановка.** Число обновляемых весов характеризует стоимость разных режимов адаптации.

Выполните следующие действия:
1. Посчитайте параметры каждого режима разморозки.
2. Проверьте порядок frozen, partial, full.

**Результат.** param_report.json с числами параметров трёх моделей.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [8]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

param_report = {}
for mode in ['frozen', 'partial', 'full']:
    m = build_model(mode=mode)
    t, tr = count_params(m)
    param_report[mode] = {'total_params': t, 'trainable_params': tr}
    print(f"Mode: {mode}, Trainable: {tr}/{t}")

with open(ARTIFACTS / "param_report.json", "w") as f:
    json.dump(param_report, f, indent=4)
check(param_report['frozen']['trainable_params'] < param_report['partial']['trainable_params'] < param_report['full']['trainable_params'], "Параметры посчитаны верно")

Mode: frozen, Trainable: 5130/11181642
Mode: partial, Trainable: 8398858/11181642
Mode: full, Trainable: 11181642/11181642
✔ Параметры посчитаны верно


### Поклассовые ошибки

*Вес: 1 балл.*

**Постановка.** Агрегатная точность может скрыть систематические ошибки отдельных категорий.

Выполните следующие действия:
1. Вычислите recall для каждого класса по test confusion matrix.
2. Найдите класс с наименьшим recall.

**Результат.** class_report.json с поклассовым recall и худшим классом.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [9]:
cm = test_cm
recall_per_class = {}
for i in range(10):
    # Проверяем, чтобы класс вообще был в матрице и чтобы сумма не была 0
    if i < cm.shape[0] and cm[i, :].sum() > 0:
        recall = cm[i, i] / cm[i, :].sum()
    else:
        recall = 0.0 # Если класса не было в выборке, ставим 0
    recall_per_class[i] = float(recall)

worst_class = min(recall_per_class, key=recall_per_class.get)
class_report = {'recall_per_class': recall_per_class, 'worst_class': worst_class}

with open(ARTIFACTS / "class_report.json", "w") as f:
    json.dump(class_report, f, indent=4)

print(f"Худший класс: {worst_class} (Recall: {recall_per_class[worst_class]:.4f})")
check(True, "Поклассовые ошибки сохранены")

Худший класс: 5 (Recall: 0.5620)
✔ Поклассовые ошибки сохранены


### Точность и стоимость обучения

*Вес: 1 балл.*

**Постановка.** Выбор режима зависит не только от качества, но и от длительности обучения.

Выполните следующие действия:
1. Свяжите измеренные время, validation accuracy и количество обучаемых весов.
2. Сохраните сопоставимую таблицу режимов.

**Результат.** efficiency_report.json с наблюдаемыми величинами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [10]:
efficiency_report = {}
for mode in modes:
    t, tr = count_params(build_model(mode=mode))
    efficiency_report[mode] = {
        'val_f1': metrics[mode]['val_f1'],
        'time_sec': metrics[mode]['time_sec'],
        'trainable_params': tr
    }

with open(ARTIFACTS / "efficiency_report.json", "w") as f:
    json.dump(efficiency_report, f, indent=4)

print("Mode | Val F1 | Time (s) | Trainable Params")
for mode in modes:
    r = efficiency_report[mode]
    print(f"{mode} | {r['val_f1']:.4f} | {r['time_sec']:.2f} | {r['trainable_params']}")
check(True, "Отчет эффективности сохранен")

Mode | Val F1 | Time (s) | Trainable Params
frozen | 0.3335 | 68.19 | 5130
partial | 0.6442 | 69.88 | 8398858
full | 0.7482 | 77.56 | 11181642
✔ Отчет эффективности сохранен


### Анализ ошибок при максимальном сдвиге

*Вес: 1 балл.*

**Постановка.** Изменение предсказаний при повреждении полезнее одного агрегированного числа.

Выполните следующие действия:
1. Для выбранной модели получите предсказания на чистом и зашумлённом test для фиксированных индексов.
2. Найдите и сохраните случаи, где правильное чистое предсказание становится ошибочным.

**Результат.** shift_failures.json с индексами и предсказаниями на реальных изображениях.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [11]:
max_sigma = 0.5
noisy_t_max = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    AddGaussianNoise(0., max_sigma)
])
noisy_set_max = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=noisy_t_max)
noisy_loader_max = DataLoader(Subset(noisy_set_max, list(range(20))) if FAST_DEV_RUN else noisy_set_max, batch_size=64)

best_model.eval()
shift_failures = {"indices": [], "clean_pred": [], "noisy_pred": [], "true_label": []}

with torch.no_grad():
    for i, ((inputs_clean, labels), (inputs_noisy, _)) in enumerate(zip(test_loader, noisy_loader_max)):
        inputs_clean, inputs_noisy, labels = inputs_clean.to(DEVICE), inputs_noisy.to(DEVICE), labels.to(DEVICE)

        clean_out = best_model(inputs_clean)
        noisy_out = best_model(inputs_noisy)

        _, clean_pred = torch.max(clean_out, 1)
        _, noisy_pred = torch.max(noisy_out, 1)

        # Ищем где чистая была права, а зашумленная ошиблась
        failures = (clean_pred == labels) & (noisy_pred != labels)
        for j in range(len(failures)):
            if failures[j]:
                idx = i * 64 + j
                shift_failures["indices"].append(idx)
                shift_failures["clean_pred"].append(clean_pred[j].item())
                shift_failures["noisy_pred"].append(noisy_pred[j].item())
                shift_failures["true_label"].append(labels[j].item())

with open(ARTIFACTS / "shift_failures.json", "w") as f:
    json.dump(shift_failures, f, indent=4)

print(f"Найдено {len(shift_failures['indices'])} ошибок при максимальном сдвиге (чистая угадала, зашумленная облажалась).")
check(True, "Ошибки сдвига сохранены")

Найдено 4167 ошибок при максимальном сдвиге (чистая угадала, зашумленная облажалась).
✔ Ошибки сдвига сохранены


### Творческий бонус: новый тип сдвига

*Вес: 1 дополнительный балл.*

**Постановка.** Не все повреждения изображений похожи на гауссовский шум.

Выполните следующие действия:
1. Выберите воспроизводимое фотометрическое или геометрическое преобразование test-изображений.
2. Сравните точность с чистым test и приведите три конкретных примера изменения предсказания.

**Результат.** bonus_shift.json с описанием преобразования, метрикой и индексами примеров.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [12]:
# Сделаем геометрический сдвиг - повернем картинки на 30 градусов
rot_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    transforms.RandomRotation(degrees=(30, 30)) # Фиксированный поворот 30
])
rot_set = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=rot_transform)
rot_loader = DataLoader(Subset(rot_set, list(range(20))) if FAST_DEV_RUN else rot_set, batch_size=64)

rot_acc, _, _ = evaluate(best_model, rot_loader, DEVICE)

# Берем 3 примера, где изменилось предсказание
rot_failures = {"indices": [], "clean_pred": [], "rot_pred": [], "true_label": []}
best_model.eval()
with torch.no_grad():
    for i, ((inputs_clean, labels), (inputs_rot, _)) in enumerate(zip(test_loader, rot_loader)):
        inputs_clean, inputs_rot, labels = inputs_clean.to(DEVICE), inputs_rot.to(DEVICE), labels.to(DEVICE)
        _, clean_pred = torch.max(best_model(inputs_clean), 1)
        _, rot_pred = torch.max(best_model(inputs_rot), 1)
        fails = (clean_pred == labels) & (rot_pred != labels)
        for j in range(len(fails)):
            if fails[j] and len(rot_failures["indices"]) < 3:
                rot_failures["indices"].append(i * 64 + j)
                rot_failures["clean_pred"].append(clean_pred[j].item())
                rot_failures["rot_pred"].append(rot_pred[j].item())
                rot_failures["true_label"].append(labels[j].item())

bonus_shift = {
    "shift_type": "Rotation 30 degrees",
    "clean_acc": test_acc,
    "shifted_acc": rot_acc,
    "examples": rot_failures
}

with open(ARTIFACTS / "bonus_shift.json", "w") as f:
    json.dump(bonus_shift, f, indent=4)

print(f"Точность при повороте: {rot_acc:.4f}")
check(len(rot_failures["indices"]) > 0, "Найдены примеры изменения предсказания при сдвиге")

Точность при повороте: 0.4248
✔ Найдены примеры изменения предсказания при сдвиге


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.
